# The Full Transformer -- Encoder + Decoder

Attention is the star. Everything else -- residuals, normalization, feed-forward, cross-attention -- is the scaffolding that lets you stack it deep.

## Problem definition

A single attention layer is a feature extractor, not a model. One matmul perlayer is not enough capacity for language. You need depth -- and depth breaks without the right plumbing.

## Basic Concept: 

### Six pieces of transformer

1. Embedding + Positional Signal.
2. Self-Attention
3. Feed-Forward Network.
4. Residual Connection
5. Layer normalization
6. Cross-attention

### Encoder block
```
X --> LN --> MHA(self) --> + --> LN --> FFN --> + --> out
                           ^                    ^
                           |-------Residual-----|
```

Encoder is bidirectional, No masking. All positions see all positions.

### Decoder Block
```
X --> LN --> MHA(masked self) --> + --> LN --> MHA(cross to encoder) --> + --> LN --> FFN --> + --> out
```

Decoder has three sublayers per block. The middle one - cross attention -- is the only place information flows from encoder to decoder. In a pure decoder-only architecture, cross-attention is **omitted** and you just have masked self-attention + FFN

### Pre-norm VS Post-norm

1. Pre-norm  `x + sublayer(LN(x))`

2. Post-norm `LN(x + sublayer())`

Post-norm is harder to train deeply without careful warmup. Pre-norm is the 2026 default.

### Comparison

Vaswani 2017 shipped LayerNorm + ReLU. Modern stacks replaced both. What production blocks actually look like:

| Component | 2017 | 2026 |
|-----------|------|------|
| Normalization | LayerNorm | RMSNorm |
| FFN activation | ReLU | SwiGLU |
| FFN expansion | 4× | 2.6× (SwiGLU uses three matrices, total params match) |
| Position | Sinusoidal absolute | RoPE |
| Attention | Full MHA | GQA (or MLA) |
| Bias terms | Yes | No |

RMSNorm drops the mean-centering of LayerNorm (one fewer subtraction), which saves compute and is empirically at least as stable. SwiGLU (`Swish(W1 x) ⊙ W3 x`) consistently outperforms ReLU/GELU FFN by ~0.5 point ppl in the Llama, PaLM and Qwen papers.


### Parameter count

For one block with `d_model = d` and FFN expansion `r`:
* MHA: `4 * d^2` (Q, K, V, O Projections)
* FFN: `3 * d * (r * d)`
* Norms: negligible

# Build your Own

In [ ]:
import math
import random
from typing import List

class Matrix:
    __slots__ = ("rows", "cols", "data")

    def __init__(self, rows, cols, fill=0.0, data=None):
        self.rows = rows
        self.cols = cols
        self.data = data or [fill] * (rows * cols)

    def get(self, i, j):
        return self.data[i * self.cols + j]

    def set(self, i, j, value):
        self.data[i * self.cols + j] = value

    def row(self, i):
        return self.data[i * self.cols : (i + 1) * self.cols]

    def copy(self):
        return Matrix(self.rows, self.cols, data=list(self.data))


def randn(rows, cols, rng, scale=None):
    if scale is None:
        scale = math.sqrt(2.0 / (rows + cols))

    m = Matrix(rows, cols)
    for i in range(rows * cols):
        m.data[i] = rng.gauss(0.0, scale)
    return m

def matmul(A, B):
    out = Matrix(A.rows, B.cols)
    for i in range(A.rows):
        for j in range(B.cols):
            for k in range(A.cols):
                out.set(i, j, out.get(i, j) + A.get(i, k) * B.get(k, j))
    return out

def transpose(A):
    out = Matrix(A.cols, A.rows)
    for i in range(A.rows):
        for j in range(A.cols):
            out.set(j, i, A.get(i, j))
    return out

def add(A, B):
    assert A.rows == B.rows and A.cols == B.cols
    return Matrix(A.rows, A.cols, data=[a + b for a, b in zip(A.data, B.data)])

def softmax_rows(A, mask=None):
    out = Matrix(A.rows, A.cols)
    for i in range(A.rows):
        row = A.row(i)
        if mask is not None:
            row = [row[j] if not mask[i][j] else float("-inf") for j in range(A.cols)]
        m = max(v for v in row if v != float("-inf"))
        exps = [math.exp(v - m) if v!= float("-inf") else 0.0 for v in row]
        s = sum(exps)
        for j, e in enumerate(exps):
            out.set(i, j, e / s if s > 0.0 else 0.0)
    return out

def layer_norm(X, eps=1e-5):
    out = Matrix(X.rows, X.cols)
    for i in range(X.rows):
        row = X.row(i)
        mean = sum(row) / len(row)
        var = sum((v - mean) ** 2 for v in row) / len(row)
        denom = math.sqrt(var + eps)
        for j in range(X.cols):
            out.set(i, j, (row[j] - mean) / denom)
    return out

def rms_norm(X, eps=1e-6):
    out = Matrix(X.rows, X.cols)
    for i in range(X.rows):
        row = X.row(i)
        rms = math.sqrt(sum(v * v for v in row) / len(row) + eps)
        for j in range(X.cols):
            out.set(i, j, row[j] / rms)
    return out

def silu(x):
    return x / (1.0 + math.exp(-x))


def ffn_swiglu(X, W1, W2, W3):
    h1 = matmul(X, W1)
    h3 = matmul(X, W3)
    gated = Matrix(h1.rows, h1.cols)
    for i in range(len(h1.data)):
        gated.data[i] = silu(h1.data[i]) * h3.data[i]

    return matmul(gated, W2)

def ffn_relu(X, W1, W2):
    h = matmul(X, W1)
    for i in range(len(h.data)):
        if h.data[i] < 0.0:
            h.data[i] = 0.0
    return matmul(h, W2)

def scaled_dot_product_attention(Q, K, V, causal=False):
    dk = Q.cols
    scores = matmul(Q, transpose(K))
    inv = 1.0 / math.sqrt(dk)
    for i in range(len(scores.data)):
        scores.data[i] *= inv
    mask = None
    if causal:
        mask = [[j > i for j in range(scores.cols)] for i in range(scores.rows)]
    w = softmax_rows(scores, mask)
    return matmul(w, V)

def multi_head_attention(X, Wq, Wk, Wv, Wo, n_heads, causal=False, kv_source=None):
    Q = matmul(X, Wq)
    kv_input = kv_source if kv_source is not None else X
    K = matmul(kv_input, Wk)
    V = matmul(kv_input, Wv)

    d_head = Q.cols // n_heads
    head_outs = []
    for h in range(n_heads):
        Qh = Matrix(Q.rows, d_head,
            data=[Q.get(i, h * d_head + j) for i in range(Q.rows) for j in range(d_head)]
        )
        Kh = Matrix(K.rows, d_head,
            data=[K.get(i, h * d_head + j) for i in range(K.rows) for j in range(d_head)]
        )
        Vh = Matrix(V.rows, d_head,
            data=[V.get(i, h * d_head + j) for i in range(V.rows) for j in range(d_head)]
        )
        head_outs.append(scaled_dot_product_attention(Qh, Kh, Vh, causal))
    concat = Matrix(X.rows, X.cols)
    for h, H in enumerate(head_outs):
        for i in range(H.rows):
            for j in range(d_head):
                concat.set(i, h * d_head + j, H.get(i, j))
    return matmul(concat, Wo)

class BlockParams:
    def __init__(self, d, n_heads, ffn_expansion, rng, use_swiglu=True):
        self.d = d
        self.n_heads = n_heads
        self.use_swiglu = use_swiglu
        self.Wq = randn(d, d, rng)
        self.Wk = randn(d, d, rng)
        self.Wv = randn(d, d, rng)
        self.Wo = randn(d, d, rng)
        h = int(d * ffn_expansion)
        if use_swiglu:
            self.W1 = randn(d, h, rng)
            self.W2 = randn(h, d, rng)
            self.W3 = randn(d, h, rng)
        else:
            self.W1 = randn(d, h, rng)
            self.W2 = randn(h, d, rng)

        # Cross-attention (Decoder)
        self.Wq_x = randn(d, d, rng)
        self.Wk_x = randn(d, d, rng)
        self.Wv_x = randn(d, d, rng)
        self.Wo_x = randn(d, d, rng)

def encoder_block(x, p):
    # pre-norm self-attention + residual
    h = rms_norm(x)
    a = multi_head_attention(h, p.Wq, p.Wk, p.Wv, p.Wo, p.n_heads)
    x = add(x, a)
    # pre-norm FFN + residual
    h = rms_norm(x)
    f = ffn_swiglu(h, p.W1, p.W2, p.W3) if p.use_swiglue else ffn_relu(h, p.W1, p.W2)
    x = add(x, f)
    return x

def decoder_block(x, encoder_output, p):
    # masked self-attention
    h = rms_norm(x)
    a = multi_head_attention(h, p.Wq, p.Wk, p.Wv, p.Wo, p.n_heads, causal=True)
    x = add(x, a)
    # cross-attention to encoder output
    h = rms_norm(x)
    a = multi_head_attention(h, p.Wq_x, p.Wk_x, p.Wv_x, p.Wo_x, p.n_heads, causal=False, kv_source=encoder_output)
    x = add(x, a)
    # pre-norm FFN
    h = rms_norm(x)
    f = ffn_swiglu(h, p.W1, p.W2, p.W3) if p.use_swiglue else ffn_relu(h, p.W1, p.W2)
    x = add(x, f)
    return x


In [ ]:
### Cross-attention：`kv_source` 是什么？

在 `decoder_block` 的 cross-attention 里：

```python
multi_head_attention(h, p.Wq_x, p.Wk_x, p.Wv_x, p.Wo_x, ...,
                     causal=False, kv_source=encoder_output)
```

**`kv_source` = encoder 最后一层的输出 `encoder_output`**，是 decoder 唯一读取源句信息的地方。

---

#### 数据从哪来

```
源句 tokens  →  Embedding  →  encoder_block × L  →  encoder_output
                                                          │
目标句 tokens  →  Embedding  →  decoder_block  ──────────┘ (cross-attention)
```

---

#### Shape 约定（`seq_len × d_model`）

设 encoder 长度 `n_enc`，decoder 长度 `n_dec`，维度 `d`：

| 张量 | Shape | 角色 |
|------|-------|------|
| `x`（decoder 隐状态 `h`） | `(n_dec, d)` | 只投影 **Q** |
| `encoder_output`（`kv_source`） | `(n_enc, d)` | 投影 **K** 和 **V** |
| attention scores | `(n_dec, n_enc)` | 第 `i` 行：decoder 第 `i` 个 token 对所有 encoder 位置的权重 |
| 输出 | `(n_dec, d)` | 写回 decoder，shape 不变 |

代码里 `multi_head_attention` 的逻辑：

```
Q = X         @ Wq     # X 来自 decoder
K = kv_source @ Wk     # kv_source 来自 encoder
V = kv_source @ Wv
```

若 `kv_source=None`（self-attention），则 `K、V` 也从 `X` 投影，scores 为 `(n, n)` 方阵。

---

#### 和上下两层 attention 的对比

| 子层 | Q 来源 | K/V 来源 | `causal` | scores shape |
|------|--------|----------|----------|--------------|
| Masked self-attention | decoder `x` | decoder `x` | `True` | `(n_dec, n_dec)` |
| **Cross-attention** | decoder `h` | **encoder `encoder_output`** | `False` | `(n_dec, n_enc)` |

- **Self-attention**：序列内部互看；decoder 用 causal mask，不能偷看未来。
- **Cross-attention**：decoder 的每个位置可以 attend **整个** encoder 序列（翻译里「生成这个词时看源句哪些词」）。

---

#### 为何单独传 `kv_source`？

Q 和 K/V 来自**不同序列、不同长度**，不能共用同一个 `X`。`kv_source` 参数把 cross-attention 和 self-attention 复用在同一个 `multi_head_attention` 函数里：有 `kv_source` 就是 cross-attention，没有就是 self-attention。